# Segmentation and Tracking for Control and Experimental Series

This notebook builds a PyDIP-based segmentation and tracking pipeline for the Control Series and the Experimental Series.

- **Segmentation procedure:** smoothing + Otsu thresholding + morphological opening/closing + hole filling + area opening + border-object removal + seeded watershed.
- **Selection:** choose the 15 largest segmented objects in the initial frame to serve as the tracked cells.
- **Tracking criterion:** a cell in the next frame is matched to the current frame by minimal centroid displacement, with size consistency as a secondary check.
- **Validation:** manually inspect the traces of 5 selected cells and visualize their trajectories to confirm the algorithm is tracking the correct objects.


In [ ]:
from pathlib import Path
import numpy as np
import tifffile as tiff
import diplib as dip
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

img_dir = Path("Path_to_your_images")  # <-- UPDATE THIS PATH
out_dir = img_dir / "output"
out_dir.mkdir(exist_ok=True)

arr0 = tiff.imread(str(img_dir / "MTLn3-ctrl0000.tif"))

# ---- diagnostic: print intensity percentiles to choose threshold ----
flat = arr0.flatten().astype(np.float32)
for p in [50, 70, 80, 85, 90, 95]:
    print(f"  {p}th percentile intensity: {np.percentile(flat, p):.1f}")

  50th percentile intensity: 19.0
  70th percentile intensity: 23.0
  80th percentile intensity: 29.0
  85th percentile intensity: 36.0
  90th percentile intensity: 47.0
  95th percentile intensity: 64.0


SEGMENTATION PROCEDURE
1. Convert to float and apply Gaussian smoothing (sigma=1.0) to suppress pixel noise before thresholding.
2. Threshold at the 88th intensity percentile.Motivation: Otsu thresholding fails here because ~88% of pixels are dark background; it picks a threshold in the low 20s which captures dim cytoplasm and bridges touching cells. The 88th percentile (~40-50 intensity units) isolates only the bright nucleus cores, which are naturally separated even when cells are touching.
3. FillHoles to remove internal holes from thresholded nuclei.
4. BinaryAreaOpening (100 px) to remove small noise fragments.
5. Remove border-touching objects (incomplete cells at edges).
6. Euclidean distance transform to model cell interior depth.
7. Smooth distance map (sigma=1.5) before finding maxima to suppress spurious local peaks caused by jagged boundaries, ensuring one seed per nucleus.
8. SeededWatershed on the original distance map constrained to the binary mask - separates touching nuclei at their contact.
9. Shape-based filtering: reject objects outside the 20-80th size percentile, with roundness < 0.65 or aspect ratio > 2.0, to exclude debris and any remaining merged pairs.
10. Select the 15 largest remaining valid objects as tracked cells.

In [ ]:

from pathlib import Path
import numpy as np
import tifffile as tiff
import diplib as dip
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Paths ---
img_dir = Path("Path_to_your_images")  # <-- UPDATE THIS PATH
out_dir = img_dir / "output"
out_dir.mkdir(exist_ok=True)

# --- Load first frame ---
arr0 = tiff.imread(str(img_dir / "MTLn3-ctrl0000.tif"))


# Step 1-2: smoothing + percentile threshold
frame_img = dip.Image(arr0)
if frame_img.TensorElements() > 1:
    frame_img = dip.RGBToGray(frame_img)
frame_img = dip.Convert(frame_img, 'SFLOAT')

smooth = dip.Gauss(frame_img, sigmas=[1.0])
threshold = float(np.percentile(np.asarray(smooth), 88))
binary = smooth > threshold

# Step 3-4: morphological cleanup
binary = dip.FillHoles(binary)
binary = dip.BinaryAreaOpening(binary, filterSize=100)

# Step 5: remove border-touching objects
border_labels = dip.Label(binary, connectivity=1)
border_arr = np.asarray(border_labels)
border_ids = np.unique(np.concatenate([
    border_arr[0, :], border_arr[-1, :],
    border_arr[:, 0], border_arr[:, -1]
])).tolist()
for bid in border_ids:
    if bid > 0:
        binary[border_labels == int(bid)] = False

# Step 6-8: distance transform + seeded watershed
dist = dip.EuclideanDistanceTransform(binary)
dist_smooth = dip.Gauss(dist, sigmas=[1.5])
seeds = dip.Maxima(dist_smooth)
seeds = dip.Label(seeds)
labels_full = dip.SeededWatershed(dist, seeds, binary, connectivity=1, flags={'labels'})
labels_arr = np.asarray(labels_full)

# Step 9: measure shape features and filter
stats = dip.MeasurementTool.Measure(
    labels_full, features=['Size', 'Center', 'Roundness', 'AspectRatioFeret']
)
sizes, centers, roundness, aspect = {}, {}, {}, {}
for lid in stats['Size'].keys():
    if lid > 0:
        sizes[lid]     = float(stats['Size'][lid][0])
        centers[lid]   = np.array([float(stats['Center'][lid][1]),
                                    float(stats['Center'][lid][0])])
        roundness[lid] = float(stats['Roundness'][lid][0])
        aspect[lid]    = float(stats['AspectRatioFeret'][lid][0])

all_sizes  = np.array(list(sizes.values()))
size_lo    = np.percentile(all_sizes, 20) * 0.5
size_hi    = np.percentile(all_sizes, 80) * 2.5

valid_ids = [
    k for k in sizes
    if size_lo < sizes[k] < size_hi
    and roundness[k] > 0.65
    and aspect[k] < 2.0
]

# Step 10: select 15 largest valid cells
top15_ids = sorted(valid_ids, key=lambda k: sizes[k], reverse=True)[:15]

# =============================================================
# BUILD LABELED IMAGE (labels 1 to 15)
# =============================================================
selected_arr = np.zeros_like(labels_arr, dtype=np.uint32)
for new_id, old_id in enumerate(top15_ids, start=1):
    selected_arr[labels_arr == old_id] = new_id

dip.ImageWrite(dip.Image(selected_arr), str(out_dir / "control_frame0_labels.tif"))

# =============================================================
# FIGURE 1 - 3-panel summary (original / binary mask / labeled)
# =============================================================
def normalize_u8(a):
    a = a.astype(np.float32)
    lo, hi = a.min(), a.max()
    return ((a - lo) / (hi - lo) * 255).astype(np.uint8) if hi > lo else np.zeros_like(a, dtype=np.uint8)

cmap   = plt.get_cmap('tab20')
base   = normalize_u8(arr0)
rgb    = np.stack([base, base, base], axis=-1).astype(np.float32) / 255.0
overlay = rgb.copy()
for new_id, old_id in enumerate(top15_ids, start=1):
    mask  = selected_arr == new_id
    color = np.array(cmap((new_id - 1) % 20)[:3])
    overlay[mask] = overlay[mask] * 0.3 + color * 0.7

binary_arr = np.asarray(binary).astype(np.uint8) * 255

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(arr0, cmap='gray')
axes[0].set_title('Original frame 0', fontsize=12)
axes[0].axis('off')

axes[1].imshow(binary_arr, cmap='gray')
axes[1].set_title(f'Binary mask (88th percentile threshold = {threshold:.0f})', fontsize=12)
axes[1].axis('off')

axes[2].imshow(overlay)
for new_id, old_id in enumerate(top15_ids, start=1):
    c = centers[old_id]
    axes[2].text(c[1], c[0], str(new_id), color='white', fontsize=9, fontweight='bold',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.5, linewidth=0))
axes[2].set_title('15 selected labeled cells (labels 1-15)', fontsize=12)
axes[2].axis('off')

plt.suptitle('Task 1: Segmentation Procedure - Control Series Frame 0', fontsize=13)
plt.tight_layout()
plt.savefig(str(out_dir / "task1_summary.png"), dpi=150, bbox_inches='tight')
plt.close()

# =============================================================
# FIGURE 2 - standalone labeled overlay (for report figure)
# =============================================================
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(overlay, interpolation='nearest')
ax.axis('off')
ax.set_title('Control Series - Frame 0\n15 Selected and Labeled Cells', fontsize=13, pad=10)

for new_id, old_id in enumerate(top15_ids, start=1):
    c = centers[old_id]
    ax.text(c[1], c[0], str(new_id), color='white', fontsize=11, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.55, linewidth=0))

patches = [mpatches.Patch(color=cmap(i % 20), label=f'Cell {i+1}') for i in range(15)]
ax.legend(handles=patches, loc='upper right', fontsize=7, ncol=2,
          framealpha=0.8, title='Cell ID')
plt.tight_layout()
plt.savefig(str(out_dir / "task1_labeled_overlay.png"), dpi=150, bbox_inches='tight')
plt.close()

print("Task 1 complete.")
print(f"Threshold used: {threshold:.1f}")
print(f"Selected cell IDs (original): {top15_ids}")

Task 1 complete.
Threshold used: 42.5
Selected cell IDs (original): [19, 57, 61, 5, 51, 26, 77, 60, 69, 10, 78, 23, 11, 62, 4]


In [11]:
# =============================================================
# FULL LABEL IMAGE - all segmented objects
# =============================================================

# Save full label TIFF (every detected object, original IDs)
dip.ImageWrite(labels_full, str(out_dir / "control_frame0_full_labels.tif"))

# Figure: all objects colored - normalize so all labels get distinct colors
fig, ax = plt.subplots(figsize=(8, 8))

# Randomly shuffle label colors so adjacent cells are visually distinct
label_display = labels_arr.copy().astype(np.float32)
unique_ids = np.unique(labels_arr[labels_arr > 0])
shuffled = unique_ids.copy()
np.random.seed(42)
np.random.shuffle(shuffled)
label_display_shuffled = np.zeros_like(labels_arr, dtype=np.float32)
for orig, shuf in zip(unique_ids, shuffled):
    label_display_shuffled[labels_arr == orig] = shuf

# Normalize to 0-1 range for colormap
label_norm = label_display_shuffled / label_display_shuffled.max()

# Use a large cyclic colormap
colored = plt.get_cmap('nipy_spectral')(label_norm)
colored[labels_arr == 0] = [0, 0, 0, 1]  # background stays black

ax.imshow(colored, interpolation='nearest')
ax.set_title(f'All segmented objects - Frame 0  (n = {len(unique_ids)})', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(str(out_dir / "task1_full_labels.png"), dpi=150, bbox_inches='tight')
plt.close()

print(f"Total segmented objects: {len(unique_ids)}")
print("Saved: control_frame0_full_labels.tif  and  task1_full_labels.png")

Total segmented objects: 76
Saved: control_frame0_full_labels.tif  and  task1_full_labels.png
